# Coding Assignment 1
## Question 1 — Trace values and notebook state

### My initial predictions

1. First print: [1000, 800, 0]
I think "append" will only apply to "saved_incomes" and "print" will give me initial dataset.
2. Second print:600 3
I think saved_incomes[-1] will return 600 because -1 refers to the last element, and 600 was appended to that list. I expect len(incomes) to return 3 because I assume only saved_incomes has gained an additional element.
3. Third print:20%I expect first_share to equal (1000 - 800) / 1000 = 0.2. Python calculates this before incomes[0] changes to 1200, so I do not expect the later change to affect the result. The formatting :.0% displays 0.2 as 20%, without decimal places.

In [1]:
incomes = [1000, 800, 0]
saved_incomes = incomes
first_share = (incomes[0] - 800) / incomes[0]
incomes[0] = 1200
saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0, 600]
600 4
20%


### Question 1(a–b): Actual outputs and explanation
The actual outputs are:
[1200, 800, 0, 600]
600 4
20%
My initial prediction was partly incorrect because I assumed that saved_incomes = incomes created an independent list.

In reality, both variables refer to the same list in memory. Therefore, modifying incomes[0] or appending an element through saved_incomes affects the list accessed through either variable. Consequently, len(incomes) returns 4 rather than 3.

However, my prediction about first_share was correct. Python calculates and stores the value 0.2 before the income changes. Changing the list afterward does not automatically recalculate this variable. The final output is therefore still 20%.

In [2]:

# Question 1(c) — Independent copy and recalculated saving share

# Define the original list.
incomes = [1000, 800, 0]

# Create an independent copy of the original list.
saved_incomes = incomes.copy()

# Calculate the initial saving share.
first_share = (incomes[0] - 800) / incomes[0]

# Change the first income from 1000 to 1200.
incomes[0] = 1200

# Recalculate the saving share using the updated income.
first_share = (incomes[0] - 800) / incomes[0]

# Append 600 to the independent copy.
saved_incomes.append(600)

# Print the same three output lines as in the original exercise.
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")


[1200, 800, 0]
600 3
33%


### Question 1(c): Independent copy and recalculated saving share

I replaced saved_incomes = incomes with saved_incomes = incomes.copy(). This creates an independent list rather than a second reference to the same list.

As a result, appending 600 to saved_incomes no longer changes incomes. The original income list therefore contains three elements, while the copied list contains four.

I also recalculated first_share after changing incomes[0] from 1000 to 1200. The updated saving share is (1200 - 800) / 1200 = 0.3333..., displayed as 33%.

## Question 2 — Implement the policy as functions

The following three functions calculate the saving share before government support, the government transfer, and the saving share after support.

In [3]:

# QUESTION 2 — IMPLEMENTING THE POLICY AS FUNCTIONS


# -----------------------------------------------------
# 2(a) SAVING SHARE
# -----------------------------------------------------

def saving_share(income, consumption):

    # Reject negative amounts before doing any calculations.
    if income < 0 or consumption < 0:
        raise ValueError("Income and consumption cannot be negative")

    # A saving share is undefined if income is zero.
    # None is different from a saving share of 0.
    if income == 0:
        return None

    # Calculate saving as a fraction of income.
    # Negative saving shares are allowed.
    return (income - consumption) / income



# -----------------------------------------------------
# 2(b) GOVERNMENT TRANSFER
# -----------------------------------------------------

def transfer(income, consumption, size):

    # Reject invalid values before checking eligibility.
    if income < 0 or consumption < 0 or size <= 0:
        raise ValueError("Invalid income, consumption, or household size")

    # Households earning more than 600 EUR per person
    # are not eligible for support.
    # Exactly 600 EUR per person is still eligible.
    if income / size > 600:
        return 0

    # Calculate the consumption shortfall.
    # max() ensures that the shortfall cannot be negative.
    shortfall = max(consumption - income, 0)

    # The maximum transfer is 100 EUR per household member.
    cap = 100 * size

    # Transfer cannot exceed either the shortfall or the cap.
    return min(shortfall, cap)



# -----------------------------------------------------
# 2(c) SAVING SHARE AFTER GOVERNMENT SUPPORT
# -----------------------------------------------------

def post_policy_share(income, consumption, size):

    # Calculate support using the existing transfer function.
    amount = transfer(income, consumption, size)

    # Add government support to the original income.
    new_income = income + amount

    # Reuse saving_share() with the updated income.
    # Consumption remains unchanged.
    return saving_share(new_income, consumption)


### Question 2 — Design explanation

The names `income`, `consumption`, and `size` in the function definitions are parameters, while the actual values supplied when calling functions are arguments. Each function uses its arguments rather than relying on variables stored elsewhere in the notebook.

We use `return` instead of `print` because the calculated values must be available for further calculations. Input validation comes before division to prevent invalid calculations, including division by zero when household size is invalid.

The `post_policy_share` function reuses both existing functions rather than duplicating their formulas, keeping the calculations consistent.

## Question 3 — Build and interpret a checked report

### 3(a) Prepare inputs

In [4]:

# QUESTION 3(a) — PREPARE HOUSEHOLD DATA

household_ids = ["A01", "A02", "A03", "A04",
                 "A05", "A06", "A07", "A08"]

income_text = ["1200", "0", "1500", "700",
               "2400", "800", "0", "1800"]

consumptions = [1250, 150, 1600, 850,
                2600, 950, 0, 1400]

household_sizes = [2, 1, 3, 1,
                   4, 2, 2, 3]


# Check that all lists contain the same number of households. If they do not, stop before processing any observations.

if (len(household_ids) != len(income_text)
    or len(household_ids) != len(consumptions)
    or len(household_ids) != len(household_sizes)):

    raise ValueError("Household data lists have unequal lengths")


# Convert income strings into numbers, by doing NEW list to preserve the original income_text.

incomes = []

for value in income_text:
    incomes.append(int(value))


# Confirm the conversion.
print("Original income strings:", income_text)
print("Numeric incomes:", incomes)
print("Number of households:", len(household_ids))


Original income strings: ['1200', '0', '1500', '700', '2400', '800', '0', '1800']
Numeric incomes: [1200, 0, 1500, 700, 2400, 800, 0, 1800]
Number of households: 8


### 3(b) Calculate household results

In [5]:

# QUESTION 3(b) — HOUSEHOLD-LEVEL CALCULATIONS


# Create empty lists to store the calculated results.
# Each list will have one value per household.

transfers = []
incomes_after = []

savings_before = []
savings_after = []

shares_before = []
shares_after = []


# Go through the households one by one.

for i in range(len(household_ids)):

    # Extract the data for the current household.
    income = incomes[i]
    consumption = consumptions[i]
    size = household_sizes[i]

    # government transfer 
    support = transfer(income, consumption, size)

    # income after receiving the transfer.
    new_income = income + support

    # savings in EUR before and after support.
    saving_before = income - consumption
    saving_after = new_income - consumption

    # Calculate saving SHARES using our existing functions.
    share_before = saving_share(income, consumption)
    share_after = post_policy_share(income, consumption, size)

    # Save the results for later calculations.
    transfers.append(support)
    incomes_after.append(new_income)

    savings_before.append(saving_before)
    savings_after.append(saving_after)

    shares_before.append(share_before)
    shares_after.append(share_after)


# Print a labelled household report.
print("HOUSEHOLD REPORT")
print("----------------")


for i in range(len(household_ids)):

    # Convert defined shares to percentages for display.
    # Preserve undefined shares instead of treating them as zero.

    if shares_before[i] is None:
        before_display = "undefined"
    else:
        before_display = f"{shares_before[i]:.2%}"

    if shares_after[i] is None:
        after_display = "undefined"
    else:
        after_display = f"{shares_after[i]:.2%}"

    # Print one labelled line per household.
    print(
        f"{household_ids[i]} | "
        f"Income: €{incomes[i]:.2f} | "
        f"Consumption: €{consumptions[i]:.2f} | "
        f"Size: {household_sizes[i]} | "
        f"Transfer: €{transfers[i]:.2f} | "
        f"Saving before: €{savings_before[i]:.2f} | "
        f"Saving after: €{savings_after[i]:.2f} | "
        f"Share before: {before_display} | "
        f"Share after: {after_display}"
    )


HOUSEHOLD REPORT
----------------
A01 | Income: €1200.00 | Consumption: €1250.00 | Size: 2 | Transfer: €50.00 | Saving before: €-50.00 | Saving after: €0.00 | Share before: -4.17% | Share after: 0.00%
A02 | Income: €0.00 | Consumption: €150.00 | Size: 1 | Transfer: €100.00 | Saving before: €-150.00 | Saving after: €-50.00 | Share before: undefined | Share after: -50.00%
A03 | Income: €1500.00 | Consumption: €1600.00 | Size: 3 | Transfer: €100.00 | Saving before: €-100.00 | Saving after: €0.00 | Share before: -6.67% | Share after: 0.00%
A04 | Income: €700.00 | Consumption: €850.00 | Size: 1 | Transfer: €0.00 | Saving before: €-150.00 | Saving after: €-150.00 | Share before: -21.43% | Share after: -21.43%
A05 | Income: €2400.00 | Consumption: €2600.00 | Size: 4 | Transfer: €200.00 | Saving before: €-200.00 | Saving after: €0.00 | Share before: -8.33% | Share after: 0.00%
A06 | Income: €800.00 | Consumption: €950.00 | Size: 2 | Transfer: €150.00 | Saving before: €-150.00 | Saving after: €

### 3(c) Compare totals and averages

In [6]:

# QUESTION 3(c) — TOTALS AND AVERAGES

# 1. TOTAL TRANSFER AND HOUSEHOLD COUNTS

total_transfer = sum(transfers)

receiving_support = 0
negative_before = 0
negative_after = 0


for i in range(len(household_ids)):

    if transfers[i] > 0:
        receiving_support += 1

    if savings_before[i] < 0:
        negative_before += 1

    if savings_after[i] < 0:
        negative_after += 1


print("TOTALS AND COUNTS")

print(f"Total transfer: €{total_transfer:.2f}")

print(f"Households receiving support: "
      f"{receiving_support}/{len(household_ids)}")

print(f"Negative saving before: "
      f"{negative_before}/{len(household_ids)}")

print(f"Negative saving after: "
      f"{negative_after}/{len(household_ids)}")


# 2. POOLED SAVING SHARES
# Divide total saving by total income.

pooled_before = sum(savings_before) / sum(incomes)
pooled_after = sum(savings_after) / sum(incomes_after)

print("\nPOOLED SAVING SHARES")
print(f"Before: {pooled_before:.2%}")
print(f"After:  {pooled_after:.2%}")


# 3. ARITHMETIC MEANS OF DEFINED HOUSEHOLD SHARES
# Exclude None values, but retain valid zeros and negatives.

defined_before = []
defined_after = []

# Additional list for a fixed-sample comparison.
after_same_households = []


for i in range(len(household_ids)):

    if shares_before[i] is not None:
        defined_before.append(shares_before[i])

        # Keep the after-policy share for this same household.
        after_same_households.append(shares_after[i])

    if shares_after[i] is not None:
        defined_after.append(shares_after[i])


mean_before = sum(defined_before) / len(defined_before)
mean_after = sum(defined_after) / len(defined_after)

mean_after_same = (
    sum(after_same_households) / len(after_same_households)
)


print("\nMEAN HOUSEHOLD SAVING SHARES")

print(f"Before: {mean_before:.2%} "
      f"({len(defined_before)} households)")

print(f"After:  {mean_after:.2%} "
      f"({len(defined_after)} households)")


# 4. COMPARE ONLY HOUSEHOLDS DEFINED IN BOTH PERIODS

print("\nSAME-HOUSEHOLD COMPARISON")

print(f"Before: {mean_before:.2%}")
print(f"After:  {mean_after_same:.2%}")

print(f"Households included in both periods: "
      f"{len(after_same_households)}")


TOTALS AND COUNTS
Total transfer: €600.00
Households receiving support: 5/8
Negative saving before: 6/8
Negative saving after: 2/8

POOLED SAVING SHARES
Before: -4.76%
After:  2.22%

MEAN HOUSEHOLD SAVING SHARES
Before: -6.19% (6 households)
After:  -7.03% (7 households)

SAME-HOUSEHOLD COMPARISON
Before: -6.19%
After:  0.13%
Households included in both periods: 6


The pooled saving share increases from −4.76% to 2.22%, as aggregate saving rises from −€400 to €200. However, the arithmetic mean of defined household saving shares falls from −6.19% (6 households) to −7.03% (7 households).

These measures differ because the pooled share weights households by income, while the arithmetic mean gives equal weight to each included household.

Household A02 initially has an undefined share because its income is zero. After receiving support, its share becomes −50%, so it enters the post-policy mean. A07 remains undefined.

Restricting the comparison to the original six households produces a post-policy mean of 0.13%. Comparing averages across different sets of households can therefore be misleading.

### 3(d) Sort defined shares using lambda

In [7]:

# QUESTION 3(d) — SORT SAVING SHARES BY DISTANCE


# Create a new list containing only defined pre-policy shares.
# Do not modify the original shares_before list.

defined_pre_shares = []

for share in shares_before:

    if share is not None:
        defined_pre_shares.append(share)


# Sort by absolute distance from the target saving share of 10%.
# The lambda receives one saving share and calculates its distance.

sorted_shares = sorted(
    defined_pre_shares,
    key=lambda share: abs(share - 0.10)
)


# Print the resulting list of original saving shares.
print("Shares ordered by distance from 10%:")
print(sorted_shares)

# Confirm that the original list was preserved.
print("\nOriginal shares:")
print(shares_before)


Shares ordered by distance from 10%:
[0.2222222222222222, -0.041666666666666664, -0.06666666666666667, -0.08333333333333333, -0.1875, -0.21428571428571427]

Original shares:
[-0.041666666666666664, None, -0.06666666666666667, -0.21428571428571427, -0.08333333333333333, -0.1875, None, 0.2222222222222222]


The new list excludes undefined pre-policy saving shares but retains all valid numerical values, including negative shares. The `lambda` function receives one saving share and returns its absolute distance from the target value of 0.10. The `sorted()` function uses these distances to order the original shares from nearest to farthest. The output contains the original signed saving shares, not their distances. The original result list remains unchanged.

### 3(e) Counterfactual income change

In [8]:

# QUESTION 3(e) — COUNTERFACTUAL INCOME CHANGE


# Create an independent copy of our numeric income list.
# This preserves the original dataset.

counterfactual_incomes = incomes.copy()


# Change only A01's income from 1200 to 1202.
counterfactual_incomes[0] = 1202


# Get the original and counterfactual income values.
original_income = incomes[0]
counterfactual_income = counterfactual_incomes[0]


# Consumption and household size remain unchanged.
consumption_a01 = consumptions[0]
size_a01 = household_sizes[0]


# Calculate the original situation.
original_transfer = transfer(
    original_income, consumption_a01, size_a01
)

original_income_after = original_income + original_transfer
original_saving_after = original_income_after - consumption_a01


# Calculate the counterfactual situation.
new_transfer = transfer(
    counterfactual_income, consumption_a01, size_a01
)

new_income_after = counterfactual_income + new_transfer
new_saving_after = new_income_after - consumption_a01


# Calculate the changes between the two scenarios.
change_transfer = new_transfer - original_transfer
change_income_after = new_income_after - original_income_after
change_saving_after = new_saving_after - original_saving_after


# Print original and counterfactual results.

print("COUNTERFACTUAL: HOUSEHOLD A01")

print(f"Original income: €{original_income:.2f}")
print(f"Counterfactual income: €{counterfactual_income:.2f}")

print(f"\nOriginal transfer: €{original_transfer:.2f}")
print(f"New transfer: €{new_transfer:.2f}")
print(f"Change in transfer: €{change_transfer:+.2f}")

print(f"\nOriginal income after support: €{original_income_after:.2f}")
print(f"New income after support: €{new_income_after:.2f}")
print(f"Change in income after support: €{change_income_after:+.2f}")

print(f"\nOriginal saving after support: €{original_saving_after:.2f}")
print(f"New saving after support: €{new_saving_after:.2f}")
print(f"Change in saving after support: €{change_saving_after:+.2f}")


# Verify that the original dataset was not modified.
assert incomes[0] == 1200

print("\nOriginal income list preserved:", incomes)


COUNTERFACTUAL: HOUSEHOLD A01
Original income: €1200.00
Counterfactual income: €1202.00

Original transfer: €50.00
New transfer: €0.00
Change in transfer: €-50.00

Original income after support: €1250.00
New income after support: €1202.00
Change in income after support: €-48.00

Original saving after support: €0.00
New saving after support: €-48.00
Change in saving after support: €-48.00

Original income list preserved: [1200, 0, 1500, 700, 2400, 800, 0, 1800]


Higher original income does not necessarily produce higher income after support under this policy. At €1,200, A01 earns exactly €600 per person and qualifies for a €50 transfer, giving €1,250 after support. Increasing income to €1,202 makes income per person €601, exceeding the eligibility threshold. The transfer falls to zero, reducing income after support to €1,202. Consequently, despite earning €2 more, the household has €48 less after support and its saving falls by €48.

## Question 4 — Audit an agent-written function

### 4(a) Identify four defects

In [9]:

# We preserve deliberately broken function so we can compare its results with our repair.

def proposed_mean(incomes, consumptions):

    total = 0
    count = 0

    for income, consumption in zip(incomes, consumptions):

        share = saving_share(income, consumption)

        if share is not None and share != 0:
            total = 0
            total = total + share
            count = count + 1

    return total / count


4(a) Four defects in the proposed function:

1) Unequal list lengths are not checked - the function uses zip(incomes, consumptions) without first checking whether the two lists have equal lengths. If one list is shorter, zip() silently stops at the end of that list and ignores the remaining observations. The required behaviour is to raise ValueError.

2) Valid zero saving shares are excluded - the condition share is not None and share != 0 excludes saving shares equal to zero. However, a saving share of zero is valid and must be included in the arithmetic mean. Only None values should be excluded.

3) The running total is repeatedly reset - the statement total = 0 appears inside the loop. Every time a valid nonzero share is encountered, the previous sum is erased. Consequently, the final result divides the last included share by the total count instead of averaging all included shares.

4) Division by zero is not handled - when no saving share is included, count remains zero. The final statement return total / count then raises ZeroDivisionError. The function should instead return None when no defined shares exist, including for empty input lists.

### 4(b) Repaired function

In [10]:

# QUESTION 4(b) — REPAIRED MEAN FUNCTION

def mean_defined_share(incomes, consumptions):

    # FIX 1: Check list lengths before processing.
    # Without this, zip() could silently skip observations.
    if len(incomes) != len(consumptions):
        raise ValueError("Income and consumption lists must have the same length")

    # Initialise the total and count once, before the loop.
    total = 0
    count = 0

    # Process each income-consumption pair.
    for income, consumption in zip(incomes, consumptions):

        # Reuse the saving_share function from Question 2.
        share = saving_share(income, consumption)

        # FIX 2: Include all defined shares, including zero
        # and negative values. Exclude only None.
        if share is not None:

            # FIX 3: Add to the existing total without resetting it.
            total = total + share
            count = count + 1

    # FIX 4: Avoid division by zero.
    # If no defined shares exist, return None.
    if count == 0:
        return None

    # Return the arithmetic mean of all defined shares.
    return total / count


### 4(c) Diagnostic tests

In [11]:

# QUESTION 4(c) — DIAGNOSTIC TESTS
# These calls demonstrate the incorrect results of proposed_mean().
# Deliberately failing calls are documented separately in Markdown.


# TEST 1 — UNEQUAL LIST LENGTHS
# The proposed function ignores the unmatched second income.

print("TEST 1 — Unequal list lengths")
print("Proposed:", proposed_mean([1000, 2000], [800]))


# TEST 2 — ZERO SAVING SHARES
# Correct shares: 0.0 and 0.2
# Correct mean: (0.0 + 0.2) / 2 = 0.1

print("\nTEST 2 — Zero saving share")
print("Proposed:", proposed_mean([1000, 1000], [1000, 800]))
print("Repaired:", mean_defined_share([1000, 1000], [1000, 800]))


# TEST 3 — RESETTING THE RUNNING TOTAL
# Correct shares: 0.2 and 0.3
# Correct mean: (0.2 + 0.3) / 2 = 0.25

print("\nTEST 3 — Running total reset")
print("Proposed:", proposed_mean([1000, 1000], [800, 700]))
print("Repaired:", mean_defined_share([1000, 1000], [800, 700]))


# TEST 4 — NO DEFINED SHARES
# The repaired function correctly returns None.
# The proposed function would raise ZeroDivisionError.

print("\nTEST 4 — No defined shares")
print("Repaired:", mean_defined_share([0], [0]))


TEST 1 — Unequal list lengths
Proposed: 0.2

TEST 2 — Zero saving share
Proposed: 0.2
Repaired: 0.1

TEST 3 — Running total reset
Proposed: 0.15
Repaired: 0.25

TEST 4 — No defined shares
Repaired: None


### 4(c) Diagnostic results

The following four input pairs expose the four defects in the original function.

| Test | Incomes | Consumptions | Correct outcome | Proposed outcome |
|---|---|---|---|---|
| 1. Unequal lengths | `[1000, 2000]` | `[800]` | `ValueError` | `0.2` |
| 2. Zero saving share | `[1000, 1000]` | `[1000, 800]` | `0.1` | `0.2` |
| 3. Running total reset | `[1000, 1000]` | `[800, 700]` | `0.25` | `0.15` |
| 4. No defined shares | `[0]` | `[0]` | `None` | `ZeroDivisionError` |

**Test 1:** The proposed function silently processes only the first household because `zip()` truncates unequal input lists. The repaired function correctly raises `ValueError`.

**Test 2:** The saving shares are 0 and 0.2. Their correct mean is 0.1. The proposed function incorrectly excludes the zero share and returns 0.2. Only one nonzero share is included, avoiding interference from the running-total defect.

**Test 3:** The saving shares are 0.2 and 0.3. Their correct mean is 0.25. The proposed function resets the total during the second iteration and returns 0.3 / 2 = 0.15.

**Test 4:** The household has zero income, so its saving share is undefined. The correct mean is `None`, but the proposed function attempts to divide zero by zero.

### Deliberately failing calls

The following calls demonstrate the expected exceptions. They are recorded as Markdown rather than executable cells to ensure that the notebook runs without unhandled errors.

**Unequal lengths with the repaired function:**

```python
mean_defined_share([1000, 2000], [800])

# ValueError: Income and consumption lists must have the same length
```

**No defined shares with the original function:**

```python
proposed_mean([0], [0])

# ZeroDivisionError: division by zero
```

The repaired function also returns `None` for two empty input lists, whereas the original function would raise `ZeroDivisionError`.

In [12]:

# QUESTION 5(a) — VERIFYING OUR FUNCTIONS

import math

# math.isclose() is used for comparing floating-point numbers.


# TEST 1 — Ordinary positive saving
# €1000 income and €800 consumption gives a 20% saving share.
assert math.isclose(saving_share(1000, 800), 0.20)


# TEST 2 — Negative saving is valid
# Spending more than income produces a negative saving share.
assert math.isclose(saving_share(800, 1000), -0.25)


# TEST 3 — Zero income gives an undefined share
# None is different from the numerical value zero.
assert saving_share(0, 150) is None


# TEST 4 — Eligibility boundary
# Exactly €600 per person is eligible for support.
assert math.isclose(transfer(1200, 1250, 2), 50)


# TEST 5 — Just above the eligibility boundary
# €601 per person is not eligible.
assert transfer(1202, 1250, 2) == 0


# TEST 6 — Transfer cap
# Support is limited to €100 per household member.
assert math.isclose(transfer(0, 1000, 2), 200)


# TEST 7 — No consumption shortfall
# A household spending less than its income gets no support.
assert transfer(1000, 800, 2) == 0


# TEST 8 — Zero saving is a valid defined share
# A zero share must be included when calculating averages.
assert math.isclose(
    mean_defined_share([1000, 1000], [1000, 800]),
    0.10
)


# TEST 9 — Correct calculation of an arithmetic mean
# The shares are 0.20 and 0.30, giving a mean of 0.25.
assert math.isclose(
    mean_defined_share([1000, 1000], [800, 700]),
    0.25
)


# TEST 10 — All shares undefined
# When both incomes are zero, the mean is undefined.
assert mean_defined_share([0, 0], [100, 200]) is None


# TEST 11 — Empty input lists
# No observations also means no defined mean.
assert mean_defined_share([], []) is None


# TEST 12 — Counterfactual preserves original income data
# A01 changes only in the copied list.
assert incomes == [1200, 0, 1500, 700, 2400, 800, 0, 1800]
assert counterfactual_incomes[0] == 1202


# TEST 13 — Original income strings remain unchanged
assert income_text == ["1200", "0", "1500", "700",
                       "2400", "800", "0", "1800"]


print("All assertions passed successfully.")


All assertions passed successfully.


### Explanation of selected assertions

**Test 3 — Zero income:** Checks that the function returns `None` for an undefined saving share rather than incorrectly treating it as zero or attempting division by zero.

**Test 4 — Eligibility boundary:** Detects the mistake of using `>= 600` instead of `> 600`, which would incorrectly exclude households earning exactly €600 per person.

**Test 8 — Zero saving shares:** Detects the mistake of excluding valid zero saving shares from the arithmetic mean.

**Test 9 — Arithmetic mean:** Detects the mistake of resetting the running total inside the loop instead of accumulating all defined saving shares.